In [0]:
# MAGIC %run ../00_setup/00_adls_gen2_oauth_connection
spark.sql("""
CREATE TABLE IF NOT EXISTS sp_mobility_bronze.gps
USING DELTA
LOCATION 'abfss://bronze@stspmobilitydev001.dfs.core.windows.net/gps'
""")

In [0]:
spark.sql("SHOW TABLES IN sp_mobility_bronze").show()

In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("route_id", StringType(), True),
    StructField("agency_id", StringType(), True),
    StructField("route_short_name", StringType(), True),
    StructField("route_long_name", StringType(), True),
    StructField("route_type", IntegerType(), True)
])

gtfs_df = spark.read \
    .option("header", True) \
    .schema(schema) \
    .csv("abfss://bronze@stspmobilitydev001.dfs.core.windows.net/gtfs/")

In [0]:
display(dbutils.fs.ls("abfss://bronze@stspmobilitydev001.dfs.core.windows.net/gtfs/"))

In [0]:
gtfs_df.printSchema()
gtfs_df.show(5)

In [0]:
gtfs_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("abfss://bronze@stspmobilitydev001.dfs.core.windows.net/delta/gtfs")

In [0]:
spark.sql("""
CREATE TABLE sp_mobility_bronze.gtfs
USING DELTA
LOCATION 'abfss://bronze@stspmobilitydev001.dfs.core.windows.net/delta/gtfs'
""")